In [ ]:
!pip install -U transformers accelerate
!pip install -q transformers datasets camel-tools torch scikit-learn seaborn tqdm accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 98.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.7/124.7 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 114.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 97.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 128.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 53.0 MB/s 

In [ ]:
# =========================
# 🚀 FULL PIPELINE: PREPROCESSING → BALANCING → TRAINING → RHYTHM → INFERENCE (FAST + FIXED + METRICS)
# =========================

import re
import json
import unicodedata
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score   # ✅ ADDED
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

# =========================
# SPEED BOOST ⚡
# =========================
torch.backends.cudnn.benchmark = True

# =========================
# SEED
# =========================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("classic_16_clean_final (1).csv")
df = df.dropna(subset=["text", "poem_meter"])
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

# =========================
# 2. PREPROCESSING
# =========================
CHAR_MAP = str.maketrans({
    "أ": "ا",
    "إ": "ا",
    "آ": "ا",
    "ى": "ي",
    "ة": "ه"
})

DIACRITICS = re.compile(r'[\u064B-\u065F\u0670\u0640]')
PUNCT = re.compile(r'[^\u0621-\u064A\s]')
SPACE = re.compile(r'\s+')

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text)
    text = text.translate(CHAR_MAP)
    text = DIACRITICS.sub("", text)
    text = PUNCT.sub(" ", text)
    text = SPACE.sub(" ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)
df = df[df["clean_text"].str.len() > 0]

# =========================
# 3. RHYTHM ENGINE 🔥
# =========================
SHORT_VOWELS = "َُِ"

def text_to_rhythm(text):
    if not isinstance(text, str):
        return ""
    pattern = []
    for ch in text:
        if ch in SHORT_VOWELS:
            pattern.append("◡")
        elif ch in "اوي":
            pattern.append("—")
    return pattern

def rhythm_features(text):
    r = text_to_rhythm(text)
    if len(r) == 0:
        return [0, 0, 0]

    short = r.count("◡")
    long = r.count("—")
    total = short + long + 1e-6

    return [
        short / total,
        long / total,
        len(r)
    ]

# =========================
# 4. BALANCING
# =========================
counts = df["poem_meter"].value_counts()
max_class = counts.max()

target = int(max_class * 0.7)
min_target = int(max_class * 0.4)

def balance(df):
    parts = []
    for label, g in df.groupby("poem_meter"):
        if len(g) < min_target:
            g = g.sample(min_target, replace=True, random_state=42)
        elif len(g) > target:
            g = g.sample(target, random_state=42)
        parts.append(g)
    return pd.concat(parts).sample(frac=1, random_state=42)

df = balance(df)

# =========================
# 5. LABELING
# =========================
meters = sorted(df["poem_meter"].unique())

meter2id = {m:i for i, m in enumerate(meters)}
id2meter = {i:m for m,i in meter2id.items()}

df["label"] = df["poem_meter"].map(meter2id)

with open("labels.json", "w", encoding="utf-8") as f:
    json.dump(id2meter, f, ensure_ascii=False, indent=2)

# =========================
# 6. SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df["label"],
    test_size=0.15,
    stratify=df["label"],
    random_state=42
)

# =========================
# 7. TOKENIZER + MODEL
# =========================
model_name = "aubmindlab/bert-base-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(model_name)

MAX_LEN = 96

train_enc = tokenizer(list(X_train), truncation=True, padding=True, max_length=MAX_LEN)
test_enc  = tokenizer(list(X_test), truncation=True, padding=True, max_length=MAX_LEN)

# =========================
# 8. DATASET
# =========================
class DS(torch.utils.data.Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item

train_ds = DS(train_enc, y_train.values)
test_ds = DS(test_enc, y_test.values)

# =========================
# 9. MODEL
# =========================
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(meters)
).to(device)

model.gradient_checkpointing_enable()
model.config.use_cache = False

# =========================
# 10. METRICS ✅ ADDED
# =========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro")
    }

# =========================
# 11. FAST TRAINING ⚡
# =========================
args = TrainingArguments(
    output_dir="./best_model1",

    num_train_epochs=3,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,

    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",   # ✅ ADDED

    load_best_model_at_end=True,
    report_to="none",

    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics   # ✅ ADDED
)

trainer.train()

trainer.save_model("./best_model1")
tokenizer.save_pretrained("./best_model1")

# =========================
# 12. INFERENCE
# =========================
model.eval()

def predict(text):
    text = clean_text(text)

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LEN)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    r_feat = rhythm_features(text)
    probs = probs + (r_feat[1] * 0.05)

    pred = int(np.argmax(probs))

    return id2meter[pred], float(probs[pred])

# =========================
# 13. TEST
# =========================
tests = [
    "قِفا نَبكِ مِن ذِكرى حبيبٍ ومَنزِلِ",
    "أَلا لَيتَ الشَبابَ يَعودُ يَومًا",
    "أنا أتعلم الذكاء الاصطناعي في الجزائر"
]

for t in tests:
    m, c = predict(t)
    print("📝", t)
    print("🎯", m)
    print("📊", c)
    print("-"*50)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transfo

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.883900,0.304434,0.920869,0.924208
2,0.213100,0.191210,0.956968,0.960567
3,0.103400,0.171620,0.965749,0.968740


📝 قِفا نَبكِ مِن ذِكرى حبيبٍ ومَنزِلِ
🎯 بحر الطويل
📊 1.0220617055892944
--------------------------------------------------
📝 أَلا لَيتَ الشَبابَ يَعودُ يَومًا
🎯 بحر الوافر
📊 1.042051076889038
--------------------------------------------------
📝 أنا أتعلم الذكاء الاصطناعي في الجزائر
🎯 بحر الرمل
📊 1.015728235244751
--------------------------------------------------


In [ ]:
# =========================
# 🚀 LOAD MODEL + METER + TAF3ILAT (NO TRAINING)
# =========================

import torch
import numpy as np
import json
import re
import unicodedata
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# =========================
# LOAD MODEL
# =========================
model_path = "./best_model1"   # change if needed

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# =========================
# LOAD LABEL MAP
# =========================
with open(f"{model_path}/labels.json", "r", encoding="utf-8") as f:
    id2meter = json.load(f)

id2meter = {int(k): v for k, v in id2meter.items()}

# =========================
# CLEAN TEXT (same as training)
# =========================
CHAR_MAP = str.maketrans({
    "أ": "ا",
    "إ": "ا",
    "آ": "ا",
    "ى": "ي",
    "ة": "ه"
})

DIACRITICS = re.compile(r'[\u064B-\u065F\u0670\u0640]')
PUNCT = re.compile(r'[^\u0621-\u064A\s]')
SPACE = re.compile(r'\s+')

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text)
    text = text.translate(CHAR_MAP)
    text = DIACRITICS.sub("", text)
    text = PUNCT.sub(" ", text)
    text = SPACE.sub(" ", text).strip()
    return text

# =========================
# 16 BAHR TAF3ILAT
# =========================
meter_patterns = {
    "بحر الطويل": "فعولن مفاعيلن فعولن مفاعيلن",
    "بحر البسيط": "مستفعلن فاعلن مستفعلن فاعلن",
    "بحر الوافر": "مفاعلتن مفاعلتن فعولن",
    "بحر الكامل": "متفاعلن متفاعلن متفاعلن",
    "بحر الهزج": "مفاعيلن مفاعيلن",
    "بحر الرجز": "مستفعلن مستفعلن مستفعلن",
    "بحر الرمل": "فاعلاتن فاعلاتن فاعلاتن",
    "بحر السريع": "مستفعلن مستفعلن فاعلن",
    "بحر الخفيف": "فاعلاتن مستفعلن فاعلاتن",
    "بحر المنسرح": "مستفعلن مفعولاتُ مستفعلن",
    "بحر المديد": "فاعلاتن فاعلن فاعلاتن",
    "بحر المقتضب": "مفعولاتُ مستفعلن",
    "بحر المجتث": "مستفعلن فاعلاتن فاعلن",
    "بحر المضارع": "مفاعيلن فاعلاتن",
    "بحر المتقارب": "فعولن فعولن فعولن فعولن",
    "بحر المتدارك": "فاعلن فاعلن فاعلن فاعلن"
}

# =========================
# PREDICT FUNCTION
# =========================
def predict_meter(text):
    text = clean_text(text)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=96
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    pred_id = int(np.argmax(probs))

    meter = id2meter[str(pred_id)] if str(pred_id) in id2meter else id2meter[pred_id]
    confidence = float(probs[pred_id])

    taf3ila = meter_patterns.get(meter, "unknown taf3ilat")

    return meter, taf3ila, confidence

# =========================
# TEST
# =========================
tests = [
    "قِفا نَبكِ مِن ذِكرى حبيبٍ ومَنزِلِ",
    "أَلا لَيتَ الشَبابَ يَعودُ يَومًا",
    "أنا أتعلم الذكاء الاصطناعي في الجزائر"
]

print("\n🚀 ARABIC POETRY METER + TAF3ILAT\n")

for t in tests:
    m, p, c = predict_meter(t)

    print("📝", t)
    print("🎯 Meter:", m)
    print("📐 Taf3ilat:", p)
    print("📊 Confidence:", round(c, 3))
    print("-"*50)



🚀 ARABIC POETRY METER + TAF3ILAT

📝 قِفا نَبكِ مِن ذِكرى حبيبٍ ومَنزِلِ
🎯 Meter: بحر الطويل
📐 Taf3ilat: فعولن مفاعيلن فعولن مفاعيلن
📊 Confidence: 0.972
--------------------------------------------------
📝 أَلا لَيتَ الشَبابَ يَعودُ يَومًا
🎯 Meter: بحر الوافر
📐 Taf3ilat: مفاعلتن مفاعلتن فعولن
📊 Confidence: 0.992
--------------------------------------------------
📝 أنا أتعلم الذكاء الاصطناعي في الجزائر
🎯 Meter: بحر الرمل
📐 Taf3ilat: فاعلاتن فاعلاتن فاعلاتن
📊 Confidence: 0.966
--------------------------------------------------


## theme model

In [ ]:
# =========================
# 🔥 INSTALL
# =========================
!pip install -q transformers datasets evaluate scikit-learn pandas accelerate

# =========================
# 🔥 UPLOAD DATASET
# =========================
from google.colab import files
uploaded = files.upload()

# =========================
# 🔥 IMPORTS
# =========================
import pandas as pd
import numpy as np
import ast
import torch
import json
import os

from sklearn.utils import resample
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import evaluate

# =========================
# 🔥 LOAD DATA
# =========================
df = pd.read_csv("clean_poem_theme_final (1).csv")

def convert(v):
    try:
        return " ".join(ast.literal_eval(v))
    except:
        return ""

df["text"] = df["poem_verses"].apply(convert)
df["label"] = df["poem_theme"]

# ❌ remove "حماسة"
df = df[df["label"] != "حماسة"]

# 🔥 merge labels
df["label"] = df["label"].replace({
    "قصيدة رومنسيه": "غزل"
})

df = df.dropna()

print("📊 Distribution after cleaning:")
print(df["label"].value_counts())

# =========================
# 🔥 BALANCING
# =========================
max_size = df["label"].value_counts().max()

df_balanced = pd.concat([
    resample(df[df["label"] == label],
             replace=True,
             n_samples=max_size,
             random_state=42)
    for label in df["label"].unique()
])

df = df_balanced

print("📊 Distribution after balancing:")
print(df["label"].value_counts())

# =========================
# 🔥 ENCODING
# =========================
le = LabelEncoder()
df["label"] = le.fit_transform(df["label"])

print("Classes:", list(le.classes_))

# =========================
# 🔥 SPLIT
# =========================
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.15,
    random_state=42,
    stratify=df["label"]
)

# =========================
# 🔥 TOKENIZER
# =========================
model_name = "aubmindlab/bert-base-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_enc = tokenizer(train_texts, truncation=True, padding=True, max_length=256)
val_enc   = tokenizer(val_texts, truncation=True, padding=True, max_length=256)

# =========================
# 🔥 DATASET CLASS
# =========================
class Dataset(torch.utils.data.Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_enc, train_labels)
val_dataset   = Dataset(val_enc, val_labels)

# =========================
# 🔥 MODEL
# =========================
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(le.classes_)
)

# =========================
# 🔥 METRICS
# =========================
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=preds, references=labels)

# =========================
# 🔥 TRAINING ARGS
# =========================
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=7,
    weight_decay=0.01,
    fp16=True,
    logging_steps=50
)

# =========================
# 🔥 TRAINER
# =========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# =========================
# 🔥 TRAIN
# =========================
trainer.train()

# =========================
# 🔥 EVAL
# =========================
results = trainer.evaluate()

print("🔥 FINAL RESULTS:")
print(results)

# =========================
# 🔥 SAVE MODEL + TOKENIZER
# =========================
save_path = "arabert_poem_model1"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("✅ Model saved")

# =========================
# 🔥 SAVE LABELS (FIXED)
# =========================
os.makedirs(save_path, exist_ok=True)

labels_dict = {i: label for i, label in enumerate(le.classes_)}

with open(f"{save_path}/labels.json", "w", encoding="utf-8") as f:
    json.dump(labels_dict, f, ensure_ascii=False, indent=2)

print("✅ labels.json saved")


📊 Distribution after cleaning:
label
غزل            1237
مدح             728
رثاء            548
هجاء            261
زهد             117
قصيدة وطنيه      32
Name: count, dtype: int64
📊 Distribution after balancing:
label
زهد            1237
هجاء           1237
مدح            1237
غزل            1237
قصيدة وطنيه    1237
رثاء           1237
Name: count, dtype: int64
Classes: ['رثاء', 'زهد', 'غزل', 'قصيدة وطنيه', 'مدح', 'هجاء']


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy
1,1.106300,1.098348,0.521544
2,0.819900,0.792114,0.693896
3,0.712500,0.615304,0.757630
4,0.424700,0.524559,0.813285
5,0.279300,0.573229,0.817774
6,0.329800,0.522270,0.840215
7,0.261700,0.500753,0.861759


🔥 FINAL RESULTS:
{'eval_loss': 0.5007526874542236, 'eval_accuracy': 0.8617594254937163, 'eval_runtime': 4.3936, 'eval_samples_per_second': 253.551, 'eval_steps_per_second': 31.865, 'epoch': 7.0}
✅ Model saved
✅ labels.json saved


# fusion model rules

In [ ]:
# =========================
# 🚀 IMPORTS
# =========================
import torch
import numpy as np
import json
import re
import unicodedata
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from difflib import SequenceMatcher
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# 🔤 CLEAN TEXT (MODEL ONLY)
# =========================
CHAR_MAP = str.maketrans({
    "أ": "ا",
    "إ": "ا",
    "آ": "ا",
    "ى": "ي",
    "ة": "ه"
})

DIACRITICS = re.compile(r'[\u064B-\u065F\u0670\u0640]')
PUNCT = re.compile(r'[^\u0621-\u064A\s]')
SPACE = re.compile(r'\s+')

def clean_text_model(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text)
    text = text.translate(CHAR_MAP)
    text = DIACRITICS.sub("", text)
    text = PUNCT.sub(" ", text)
    text = SPACE.sub(" ", text).strip()
    return text


# =========================
# 🔵 LOAD METER MODEL
# =========================
meter_model_path = "./best_model1"

meter_tokenizer = AutoTokenizer.from_pretrained(meter_model_path)
meter_model = AutoModelForSequenceClassification.from_pretrained(meter_model_path)

meter_model.to(device)
meter_model.eval()

with open(f"{meter_model_path}/labels.json", "r", encoding="utf-8") as f:
    id2meter = json.load(f)

id2meter = {int(k): v for k, v in id2meter.items()}


# =========================
# 🟢 LOAD THEME MODEL
# =========================
theme_model_path = "./arabert_poem_model1"

theme_tokenizer = AutoTokenizer.from_pretrained(theme_model_path)
theme_model = AutoModelForSequenceClassification.from_pretrained(theme_model_path)

theme_model.to(device)
theme_model.eval()

with open(f"{theme_model_path}/labels.json", "r", encoding="utf-8") as f:
    id2theme = json.load(f)

id2theme = {int(k): v for k, v in id2theme.items()}


# =========================
# 📐 TAF3ILAT
# =========================
# =========================
# 16 BAHR TAF3ILAT
# =========================
meter_patterns = {
    "بحر الطويل": "فعولن مفاعيلن فعولن مفاعيلن",
    "بحر البسيط": "مستفعلن فاعلن مستفعلن فاعلن",
    "بحر الوافر": "مفاعلتن مفاعلتن فعولن",
    "بحر الكامل": "متفاعلن متفاعلن متفاعلن",
    "بحر الهزج": "مفاعيلن مفاعيلن",
    "بحر الرجز": "مستفعلن مستفعلن مستفعلن",
    "بحر الرمل": "فاعلاتن فاعلاتن فاعلاتن",
    "بحر السريع": "مستفعلن مستفعلن فاعلن",
    "بحر الخفيف": "فاعلاتن مستفعلن فاعلاتن",
    "بحر المنسرح": "مستفعلن مفعولاتُ مستفعلن",
    "بحر المديد": "فاعلاتن فاعلن فاعلاتن",
    "بحر المقتضب": "مفعولاتُ مستفعلن",
    "بحر المجتث": "مستفعلن فاعلاتن فاعلن",
    "بحر المضارع": "مفاعيلن فاعلاتن",
    "بحر المتقارب": "فعولن فعولن فعولن فعولن",
    "بحر المتدارك": "فاعلن فاعلن فاعلن فاعلن"
}


# =========================
# 🔵 PREDICT METER
# =========================
def predict_meter(text):
    text = clean_text_model(text)

    inputs = meter_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=96)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = meter_model(**inputs).logits

    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred_id = int(np.argmax(probs))

    meter = id2meter.get(pred_id) or id2meter.get(str(pred_id), "unknown")
    taf3ilat = meter_patterns.get(meter, "unknown taf3ilat")

    return meter, taf3ilat, float(probs[pred_id])


# =========================
# 🟢 PREDICT THEME
# =========================
def predict_theme(text):
    text = clean_text_model(text)

    inputs = theme_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=96)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = theme_model(**inputs).logits

    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred_id = int(np.argmax(probs))

    theme = id2theme.get(pred_id) or id2theme.get(str(pred_id), "unknown")

    return theme, float(probs[pred_id])


# =========================
# 🟣 QAFYA
# =========================
import re
from collections import Counter

# =========================
# 🔤 CLEAN + NORMALIZE
# =========================
def clean(text):
    text = re.sub(r'[\u064B-\u0652\u0670]', '', text)  # tashkeel
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)     # keep Arabic only
    text = text.strip()
    return text

def normalize(text):
    text = text.replace("أ","ا").replace("إ","ا").replace("آ","ا")
    text = text.replace("ة","ه").replace("ى","ي")
    return text

# =========================
# ✂️ LAST WORD
# =========================
def get_last_word(verse):
    words = verse.split()
    return words[-1] if words else ""

# =========================
# ✂️ REMOVE SUFFIXES
# =========================
def remove_suffix(word):
    suffixes = ["كما","هما","كم","كن","نا","ها","هم","هن","ه"]
    for s in sorted(suffixes, key=len, reverse=True):
        if word.endswith(s) and len(word) > len(s)+1:
            return word[:-len(s)]
    return word

# =========================
# 🎯 RAWI
# =========================
def extract_rawi(word):
    if not word:
        return ""
    if word[-1] in ["ا","و","ي"] and len(word) >= 2:
        return word[-2]
    return word[-1]

# =========================
# 🎯 RHYME EXTRACTION
# =========================
def extract_rhyme(verse):
    verse = normalize(clean(verse))
    last_word = remove_suffix(get_last_word(verse))

    if not last_word:
        return "", ""

    rawi = extract_rawi(last_word)

    # Rule 1: ending "اء"
    if last_word.endswith("اء"):
        return "اء", rawi

    # Rule 2: vowel + consonant endings
    if len(last_word) >= 2 and last_word[-2] in "اوي":
        return last_word[-2:], rawi

    # Rule 3: short words
    if len(last_word) <= 2:
        return rawi, rawi

    # Rule 4: normal fallback
    return rawi, rawi

# =========================
# 📜 POEM ANALYSIS
# =========================
def analyze_poem(poem):
    verses = [v.strip() for v in poem.split("\n") if v.strip()]

    rhymes = []

    for v in verses:
        rhyme, rawi = extract_rhyme(v)
        if rhyme:
            rhymes.append(rhyme)

    if not rhymes:
        return {
            "status": "❌ لا توجد قافية",
            "dominant": ("none",0),
            "ratio": 0
        }

    counter = Counter(rhymes)
    dominant, count = counter.most_common(1)[0]

    ratio = count / len(rhymes)

    if ratio == 1:
        status = "✅ قافية موحدة"
    elif ratio >= 0.66:
        status = "⚠️ شبه موحدة"
    else:
        status = "❌ غير موحدة"

    return {
        "status": status,
        "dominant": (dominant, count),
        "ratio": round(ratio,2),
        "details": dict(counter)
    }

# =========================
# 🧪 TEST
# =========================
poem = """قسوت
شدت
زينت"""

print(analyze_poem(poem))



# =========================
# 🟡 BALAGHA (FIXED STRUCTURE)
# =========================
# ==========================================================
# 🟡 BALAGHA MODULE (READY TO USE)
# ==========================================================

import re
from difflib import SequenceMatcher
from collections import Counter

# =========================
# 🔧 NORMALIZATION
# =========================
def normalize(text):
    text = re.sub(r'[ًٌٍَُِّْـ]', '', text)
    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)
    return text.strip()

def tokenize(text):
    return [w for w in normalize(text).split() if len(w) > 1]


# =========================
# 📚 DICTIONARY
# =========================
ANTONYMS = {
    "نور": ["ظلام"], "ظلام": ["نور"],
    "ليل": ["نهار"], "نهار": ["ليل"],
    "شمس": ["قمر"],

    "حياه": ["موت"], "موت": ["حياه"],
    "بقاء": ["فناء"], "فناء": ["بقاء"],

    "حب": ["كره"], "كره": ["حب"],
    "خير": ["شر"], "شر": ["خير"],
    "عدل": ["ظلم"], "ظلم": ["عدل"],
}

QURAN_PHRASES = {
    "بسم الله",
    "الحمد لله",
    "سبحان الله",
    "لا اله الا الله",
    "الله اكبر"
}

NOISE_WORDS = {"هذا", "هذه", "الذي", "التي", "الى", "على", "في", "من", "و"}


# =========================
# 🔥 1. JINAS
# =========================
def is_valid_jinas(w1, w2, sim):
    if w1 == w2:
        return False
    if w1 in NOISE_WORDS or w2 in NOISE_WORDS:
        return False
    if len(w1) < 3 or len(w2) < 3:
        return False
    if w1 in w2 or w2 in w1:
        return False
    return 0.78 <= sim < 0.95


def detect_jinas(text):
    words = tokenize(text)
    found = []
    seen = set()

    for i in range(len(words)):
        for j in range(i + 1, len(words)):
            w1, w2 = words[i], words[j]

            sim = SequenceMatcher(None, w1, w2).ratio()
            key = tuple(sorted([w1, w2]))

            if key in seen:
                continue
            seen.add(key)

            if sim >= 0.95:
                found.append({
                    "type": "جناس تام",
                    "w1": w1,
                    "w2": w2,
                    "score": round(sim, 2)
                })

            elif is_valid_jinas(w1, w2, sim):
                found.append({
                    "type": "جناس ناقص",
                    "w1": w1,
                    "w2": w2,
                    "score": round(sim, 2)
                })

    return found


# =========================
# 🔥 2. TIBAQ
# =========================
def detect_tibaq(text):
    words = set(tokenize(text))
    found = []

    for k, vals in ANTONYMS.items():
        for v in vals:
            if k in words and v in words:
                found.append({
                    "type": "طباق",
                    "a": k,
                    "b": v
                })

    return found


# =========================
# 🔥 3. MUQABALA
# =========================
def detect_muqabala(text):
    tibaq = detect_tibaq(text)

    if len(tibaq) >= 2:
        return [{
            "type": "مقابلة",
            "pairs": tibaq
        }]
    return []


# =========================
# 🔥 4. IQTIBAS
# =========================
def detect_iqtibas(text):
    clean = normalize(text)

    return [
        {"type": "اقتباس", "phrase": p}
        for p in QURAN_PHRASES
        if normalize(p) in clean
    ]


# =========================
# 🔥 5. SAJ
# =========================
def detect_saj(verses):
    endings = []

    for v in verses:
        words = tokenize(v)
        if words:
            endings.append(words[-1][-2:])

    found = []

    for i in range(len(endings) - 1):
        if endings[i] == endings[i + 1] and endings[i] not in NOISE_WORDS:
            found.append({
                "type": "سجع",
                "pair": (i + 1, i + 2),
                "ending": endings[i]
            })

    return found


# =========================
# 🔥 MAIN ANALYZER
# =========================
def analyze_balagha(text):
    verses = [v.strip() for v in text.split("\n") if v.strip()]

    result = {}

    j = detect_jinas(text)
    if j:
        result["الجناس"] = j

    t = detect_tibaq(text)
    if t:
        result["الطباق"] = t

    m = detect_muqabala(text)
    if m:
        result["المقابلة"] = m

    i = detect_iqtibas(text)
    if i:
        result["الاقتباس"] = i

    s = detect_saj(verses)
    if s:
        result["السجع"] = s

    return result


# =========================
# 🚀 FUSION
# =========================
def fusion_analysis(text):

    meter, taf, mc = predict_meter(text)
    theme, tc = predict_theme(text)
    qafya = analyze_poem(text)
    bal = analyze_balagha(text)

    return {
        "🔵 البحر": meter,
        "📐 التفعيلات": taf,
        "📊 ثقة البحر": round(mc,3),
        "🟢 الموضوع": theme,
        "📊 ثقة الموضوع": round(tc,3),
        "🟣 القافية": qafya,
        "🟡 المحسنات": bal
    }


# =========================
# 🧪 TEST
# =========================
text = """قفا نبك من ذكرى حبيب ومنزل
بسقط اللوى بين الدخول فحومل"""

print(json.dumps(fusion_analysis(text), ensure_ascii=False, indent=2))


{'status': '⚠️ شبه موحدة', 'dominant': ('ت', 2), 'ratio': 0.67, 'details': {'وت': 1, 'ت': 2}}
{
  "🔵 البحر": "بحر الطويل",
  "📐 التفعيلات": "فعولن مفاعيلن فعولن مفاعيلن",
  "📊 ثقة البحر": 0.989,
  "🟢 الموضوع": "رثاء",
  "📊 ثقة الموضوع": 0.999,
  "🟣 القافية": {
    "status": "✅ قافية موحدة",
    "dominant": [
      "ل",
      2
    ],
    "ratio": 1.0,
    "details": {
      "ل": 2
    }
  },
  "🟡 المحسنات": {}
}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [26]:
!pip install streamlit pyngrok

In [27]:
!ngrok config add-authtoken 3CR91HcRcc7PeVT2wRNi1Yr1Q8m_4ecx1Rv7N7FUdH2NBQcsK

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [50]:
%%writefile app.py
# =========================
# 🚀 STREAMLIT CONFIG (MUST BE FIRST)
# =========================
import streamlit as st
st.set_page_config(
    page_title="📜 منصة تحليل الشعر العربي",
    page_icon="✨",
    layout="wide",
    initial_sidebar_state="expanded"
)

# =========================
# 🚀 IMPORTS & AUTO-INSTALL
# =========================
import subprocess
import sys

try:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers"])
    from transformers import AutoTokenizer, AutoModelForSequenceClassification

import torch
import numpy as np
import json
import re
import unicodedata
from difflib import SequenceMatcher
from collections import Counter
import time
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# 🔤 CLEAN TEXT (MODEL ONLY)
# =========================
CHAR_MAP = str.maketrans({
    ord('أ'): ord('ا'), ord('إ'): ord('ا'), ord('آ'): ord('ا'),
    ord('ى'): ord('ي'), ord('ة'): ord('ه')
})
DIACRITICS = re.compile(r'[\u064B-\u065F\u0670\u0640]')
PUNCT = re.compile(r'[^\u0621-\u064A\s]')
SPACE = re.compile(r'\s+')

def clean_text_model(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFC", text)
    text = text.translate(CHAR_MAP)
    text = DIACRITICS.sub("", text)
    text = PUNCT.sub(" ", text)
    text = SPACE.sub(" ", text).strip()
    return text

# =========================
# 🔵 LOAD METER MODEL
# =========================
meter_model_path = "./best_model1"
meter_tokenizer = AutoTokenizer.from_pretrained(meter_model_path)
meter_model = AutoModelForSequenceClassification.from_pretrained(meter_model_path)
meter_model.to(device)
meter_model.eval()

with open(f"{meter_model_path}/labels.json", "r", encoding="utf-8") as f:
    id2meter = json.load(f)
id2meter = {int(k): v for k, v in id2meter.items()}

# =========================
# 🟢 LOAD THEME MODEL
# =========================
theme_model_path = "/content/arabert_poem_model1/arabert_poem_model1"

theme_tokenizer = AutoTokenizer.from_pretrained(theme_model_path)
theme_model = AutoModelForSequenceClassification.from_pretrained(theme_model_path)

theme_model.to(device)
theme_model.eval()

with open(f"{theme_model_path}/labels.json", "r", encoding="utf-8") as f:
    id2theme = json.load(f)

id2theme = {int(k): v for k, v in id2theme.items()}

# =========================
# 📐 TAF3ILAT & BAHR
# =========================
meter_patterns = {
    "بحر الطويل": "فعولن مفاعيلن فعولن مفاعيلن",
    "بحر البسيط": "مستفعلن فاعلن مستفعلن فاعلن",
    "بحر الوافر": "مفاعلتن مفاعلتن فعولن",
    "بحر الكامل": "متفاعلن متفاعلن متفاعلن",
    "بحر الهزج": "مفاعيلن مفاعيلن",
    "بحر الرجز": "مستفعلن مستفعلن مستفعلن",
    "بحر الرمل": "فاعلاتن فاعلاتن فاعلاتن",
    "بحر السريع": "مستفعلن مستفعلن فاعلن",
    "بحر الخفيف": "فاعلاتن مستفعلن فاعلاتن",
    "بحر المنسرح": "مستفعلن مفعولاتُ مستفعلن",
    "بحر المديد": "فاعلاتن فاعلن فاعلاتن",
    "بحر المقتضب": "مفعولاتُ مستفعلن",
    "بحر المجتث": "مستفعلن فاعلاتن فاعلن",
    "بحر المضارع": "مفاعيلن فاعلاتن",
    "بحر المتقارب": "فعولن فعولن فعولن فعولن",
    "بحر المتدارك": "فاعلن فاعلن فاعلن فاعلن"
}

def predict_meter(text):
    text = clean_text_model(text)
    inputs = meter_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=96)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = meter_model(**inputs).logits
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred_id = int(np.argmax(probs))
    meter = id2meter.get(pred_id, "unknown")
    taf3ilat = meter_patterns.get(meter, "غير معروفة")
    return meter, taf3ilat, float(probs[pred_id])

def predict_theme(text):
    if theme_tokenizer is None:
        return "غير متاح", 0.0
    text = clean_text_model(text)
    inputs = theme_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=96)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = theme_model(**inputs).logits
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pred_id = int(np.argmax(probs))
    theme = id2theme.get(pred_id, "unknown")
    return theme, float(probs[pred_id])

# =========================
# 🟣 QAFYA (RHYME) MODULE
# =========================
def clean(text):
    text = re.sub(r'[\u064B-\u0652\u0670]', '', text)
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)
    return text.strip()

def normalize1(text):
    text = text.replace("أ","ا").replace("إ","ا").replace("آ","ا")
    text = text.replace("ة","ه").replace("ى","ي")
    return text

def get_last_word(verse):
    words = verse.split()
    return words[-1] if words else ""

def remove_suffix(word):
    suffixes = ["كما","هما","كم","كن","نا","ها","هم","هن","ه"]
    for s in sorted(suffixes, key=len, reverse=True):
        if word.endswith(s) and len(word) > len(s)+1:
            return word[:-len(s)]
    return word

def extract_rawi(word):
    if not word:
        return ""
    if word[-1] in ["ا","و","ي"] and len(word) >= 2:
        return word[-2]
    return word[-1]

def extract_rhyme(verse):
    verse = normalize1(clean(verse))
    last_word = remove_suffix(get_last_word(verse))
    if not last_word:
        return "", ""
    rawi = extract_rawi(last_word)
    if last_word.endswith("اء"):
        return "اء", rawi
    if len(last_word) >= 2 and last_word[-2] in "اوي":
        return last_word[-2:], rawi
    if len(last_word) <= 2:
        return rawi, rawi
    return rawi, rawi

def analyze_poem(poem):
    verses = [v.strip() for v in poem.split("\n") if v.strip()]
    rhymes = []
    for v in verses:
        rhyme, _ = extract_rhyme(v)
        if rhyme:
            rhymes.append(rhyme)
    if not rhymes:
        return {"status": "❌ لا توجد قافية", "dominant": ("لا يوجد",0), "ratio": 0}
    counter = Counter(rhymes)
    dominant, count = counter.most_common(1)[0]
    ratio = count / len(rhymes)
    if ratio == 1:
        status = "✅ قافية موحدة"
    elif ratio >= 0.66:
        status = "⚠️ شبه موحدة"
    else:
        status = "❌ غير موحدة"
    return {"status": status, "dominant": (dominant, count), "ratio": round(ratio,2), "details": dict(counter)}

# =========================
# 🟡 BALAGHA (RHETORIC) MODULE
# =========================
def normalize_bal(text):
    text = re.sub(r'[ًٌٍَُِّْـ]', '', text)
    text = re.sub(r'[إأآا]', 'ا', text)
    text = re.sub(r'ى', 'ي', text)
    text = re.sub(r'ة', 'ه', text)
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)
    return text.strip()

def tokenize(text):
    return [w for w in normalize_bal(text).split() if len(w) > 1]

ANTONYMS = {
    "نور": ["ظلام"], "ظلام": ["نور"], "ليل": ["نهار"], "نهار": ["ليل"], "شمس": ["قمر"],
    "حياه": ["موت"], "موت": ["حياه"], "بقاء": ["فناء"], "فناء": ["بقاء"],
    "حرب": ["سلام"], "سلام": ["حرب"], "قتال": ["صلح"], "صلح": ["قتال"],
    "قوه": ["ضعف"], "ضعف": ["قوه"], "شجاعه": ["جبن"], "جبن": ["شجاعه"],
    "فرح": ["حزن"], "حزن": ["فرح"], "سعاده": ["شقاء"], "شقاء": ["سعاده"], "حب": ["كره"], "كره": ["حب"],
    "علم": ["جهل"], "جهل": ["علم"], "فهم": ["غفله"], "غفله": ["فهم"],
    "كبير": ["صغير"], "صغير": ["كبير"], "كثير": ["قليل"], "قليل": ["كثير"],
    "اول": ["اخر"], "اخر": ["اول"], "قديم": ["حديث"], "حديث": ["قديم"],
    "غني": ["فقير"], "فقير": ["غني"], "قوي": ["ضعيف"], "ضعيف": ["قوي"],
    "اعلى": ["اسفل"], "اسفل": ["اعلى"], "داخل": ["خارج"], "خارج": ["داخل"],
    "خير": ["شر"], "شر": ["خير"], "عدل": ["ظلم"], "ظلم": ["عدل"],
    "ذكاء": ["غباء"], "غباء": ["ذكاء"], "يقظه": ["نوم"], "نوم": ["يقظه"],
    "هدى": ["ضلال"], "ضلال": ["هدى"]
}
QURAN_PHRASES = {"بسم الله", "الحمد لله", "سبحان الله", "لا اله الا الله", "الله اكبر"}
NOISE_WORDS = {"هذا", "هذه", "الذي", "التي", "الى", "على", "في", "من", "و"}

def is_valid_jinas(w1, w2, sim):
    if w1 == w2 or w1 in NOISE_WORDS or w2 in NOISE_WORDS: return False
    if len(w1) < 3 or len(w2) < 3: return False
    if w1 in w2 or w2 in w1: return False
    return 0.78 <= sim < 0.95

def detect_jinas(text):
    words = tokenize(text)
    found, seen = [], set()
    for i in range(len(words)):
        for j in range(i+1, len(words)):
            w1, w2 = words[i], words[j]
            sim = SequenceMatcher(None, w1, w2).ratio()
            key = tuple(sorted([w1,w2]))
            if key in seen: continue
            seen.add(key)
            if sim >= 0.95:
                found.append({"type": "جناس تام", "w1": w1, "w2": w2, "score": round(sim,2)})
            elif is_valid_jinas(w1, w2, sim):
                found.append({"type": "جناس ناقص", "w1": w1, "w2": w2, "score": round(sim,2)})
    return found

def detect_tibaq(text):
    words = set(tokenize(text))
    found = []
    for k, vals in ANTONYMS.items():
        for v in vals:
            if k in words and v in words:
                found.append({"type": "طباق", "a": k, "b": v})
    return found

def detect_muqabala(text):
    tibaq = detect_tibaq(text)
    if len(tibaq) >= 2:
        return [{"type": "مقابلة", "pairs": tibaq}]
    return []

def detect_iqtibas(text):
    clean_txt = normalize_bal(text)
    return [{"type": "اقتباس", "phrase": p} for p in QURAN_PHRASES if normalize_bal(p) in clean_txt]

def detect_saj(verses):
    endings = []
    for v in verses:
        words = tokenize(v)
        if words:
            endings.append(words[-1][-2:])
    found = []
    for i in range(len(endings)-1):
        if endings[i] == endings[i+1] and endings[i] not in NOISE_WORDS:
            found.append({"type": "سجع", "pair": (i+1, i+2), "ending": endings[i]})
    return found

def analyze_balagha(text):
    verses = [v.strip() for v in text.split("\n") if v.strip()]
    result = {}
    j = detect_jinas(text); t = detect_tibaq(text); m = detect_muqabala(text); i = detect_iqtibas(text); s = detect_saj(verses)
    if j: result["الجناس"] = j
    if t: result["الطباق"] = t
    if m: result["المقابلة"] = m
    if i: result["الاقتباس"] = i
    if s: result["السجع"] = s
    return result

# =========================
# 🚀 FUSION ANALYSIS
# =========================
def fusion_analysis(text):
    meter, taf, mc = predict_meter(text)
    theme, tc = predict_theme(text)
    qafya = analyze_poem(text)
    bal = analyze_balagha(text)
    return {
        "🔵 البحر": meter, "📐 التفعيلات": taf, "📊 ثقة البحر": round(mc,3),
        "🟢 الموضوع": theme, "📊 ثقة الموضوع": round(tc,3),
        "🟣 القافية": qafya, "🟡 المحسنات": bal
    }

# =========================
# 🎨 PROFESSIONAL UI (ARABIC-FIRST)
# =========================
# Custom RTL-friendly CSS
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?font-family=Cairo:wght@400;700;900&display=swap');
* {
    font-family: 'Cairo', 'Segoe UI', sans-serif;
}
html, body, .stApp {
    direction: rtl;
    text-align: right;
}
.stApp {
    background: radial-gradient(circle at 10% 30%, #0f172a, #020617);
}
/* Sidebar */
.css-1d391kg, .css-163ttbj, .stSidebar {
    background: rgba(15, 25, 50, 0.9);
    backdrop-filter: blur(10px);
    border-left: 1px solid rgba(99,102,241,0.3);
}
/* Cards */
.glow-card {
    background: rgba(15, 25, 50, 0.65);
    backdrop-filter: blur(12px);
    border-radius: 28px;
    padding: 1.6rem;
    border: 1px solid rgba(99, 102, 241, 0.3);
    box-shadow: 0 8px 32px rgba(0,0,0,0.4);
    transition: all 0.3s ease;
}
.glow-card:hover {
    transform: translateY(-5px);
    border-color: #22d3ee;
    box-shadow: 0 20px 40px rgba(0,0,0,0.6), 0 0 0 2px rgba(34,211,238,0.2);
}
.metric-value {
    font-size: 2.2rem;
    font-weight: 900;
    background: linear-gradient(135deg, #e2e8f0, #a5f3fc);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}
.metric-label {
    font-size: 0.9rem;
    color: #94a3b8;
    letter-spacing: 1px;
    margin-bottom: 0.5rem;
}
/* Text area */
.stTextArea textarea {
    background-color: #0f172a !important;
    color: #f1f5f9 !important;
    border: 1px solid #334155;
    border-radius: 20px;
    font-size: 1.1rem;
    padding: 1rem;
    direction: rtl;
}
.stTextArea textarea:focus {
    border-color: #22d3ee;
    box-shadow: 0 0 0 2px rgba(34,211,238,0.3);
}
/* Buttons */
.stButton button {
    background: linear-gradient(90deg, #4f46e5, #22d3ee);
    border: none;
    border-radius: 40px;
    padding: 0.6rem 2rem;
    font-weight: bold;
    font-size: 1.2rem;
    color: white;
    transition: all 0.3s;
    width: 100%;
}
.stButton button:hover {
    transform: scale(1.02);
    box-shadow: 0 8px 25px rgba(34,211,238,0.5);
    letter-spacing: 1px;
}
/* Tabs */
.stTabs [data-baseweb="tab-list"] {
    gap: 8px;
    background: rgba(0,0,0,0.2);
    border-radius: 40px;
    padding: 0.5rem;
}
.stTabs [data-baseweb="tab"] {
    border-radius: 40px;
    padding: 0.5rem 1.5rem;
    font-weight: bold;
    color: #cbd5e1;
}
.stTabs [aria-selected="true"] {
    background: linear-gradient(90deg, #4f46e5, #22d3ee);
    color: white;
}
/* Progress bar */
.stProgress > div > div {
    background: linear-gradient(90deg, #4f46e5, #22d3ee);
}
/* Headers */
h1, h2, h3 {
    font-weight: 800;
}
</style>
""", unsafe_allow_html=True)

# Sidebar content
with st.sidebar:
    st.image("https://cdn-icons-png.flaticon.com/512/1995/1995572.png", width=80)
    st.markdown("### 🧠 منصة تحليل الشعر العربي")
    st.markdown("**أداة ذكية تعتمد على الذكاء الاصطناعي لتحليل القصائد العربية**")
    st.markdown("---")
    st.markdown("#### ✨ الخدمات المقدمة")
    st.markdown("""
    - 🔍 **البحر العروضي** والتفعيلات
    - 🎯 **تصنيف الموضوع** (مدح، رثاء، غزل، ...)
    - 🟣 **تحليل القافية** والروي
    - 🟡 **المحسنات البديعية** (جناس، طباق، سجع، اقتباس)
    """)
    st.markdown("---")
    st.markdown("#### 📌 طريقة الاستخدام")
    st.markdown("""
    1. الصق نص القصيدة (بيت كل سطر)
    2. اضغط زر **تحليل الآن**
    3. تصفح النتائج في الأقسام المختلفة
    """)
    st.markdown("---")
    st.markdown("#### ⚡ التقنيات")
    st.markdown("- AraBERT fine-tuned\n- PyTorch & Transformers\n- Streamlit")
    st.caption("الإصدار 2.0 – تصميم احترافي عصري")

# Main header
st.markdown("""
<div style="text-align: center; padding: 1rem 0;">
    <h1 style="font-size: 3.5rem; background: linear-gradient(120deg, #c084fc, #22d3ee); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">
        📜 منصة تحليل الشعر العربي
    </h1>
    <p style="color: #94a3b8; font-size: 1.2rem;">تحليل ذكي للقصائد: البحر، القافية، البلاغة، الموضوع</p>
</div>
""", unsafe_allow_html=True)

# Input area
poem_input = st.text_area(
    "✍️ أدخل القصيدة (بيت كل سطر)",
    height=250,
    placeholder="مثال:\nألا هبي بصحنك فاصبحينا\nفلا تبقي خمر الأندلسينا\nوصافيها لنا يا ساقيينا\nفقد هام الفؤاد فما يسلينا"
)

# Example poem button
col_ex, _ = st.columns([1,4])
with col_ex:
    if st.button("📖 قصيدة تجريبية"):
        poem_input = """ألا هبي بصحنك فاصبحينا
فلا تبقي خمر الأندلسينا
وصافيها لنا يا ساقيينا
فقد هام الفؤاد فما يسلينا"""
        st.rerun()

# Main analysis button
if st.button("🚀 تحليل الآن", use_container_width=True):
    if not poem_input.strip():
        st.warning("⚠️ الرجاء إدخال نص القصيدة")
    else:
        with st.spinner("✨ جاري التحليل الذكي ..."):
            time.sleep(0.3)
            result = fusion_analysis(poem_input)

        st.toast("تم التحليل بنجاح!", icon="✅")

        # Results header
        st.markdown("## 📊 النتائج الرئيسية")
        col1, col2, col3 = st.columns(3)

        with col1:
            st.markdown('<div class="glow-card">', unsafe_allow_html=True)
            st.markdown('<div class="metric-label">🔵 البحر العروضي</div>', unsafe_allow_html=True)
            st.markdown(f'<div class="metric-value">{result["🔵 البحر"]}</div>', unsafe_allow_html=True)
            st.progress(result["📊 ثقة البحر"], text=f"نسبة الثقة: {result['📊 ثقة البحر']*100:.1f}%")
            st.markdown('</div>', unsafe_allow_html=True)

        with col2:
            st.markdown('<div class="glow-card">', unsafe_allow_html=True)
            st.markdown('<div class="metric-label">🟢 الموضوع</div>', unsafe_allow_html=True)
            st.markdown(f'<div class="metric-value">{result["🟢 الموضوع"]}</div>', unsafe_allow_html=True)
            st.progress(result["📊 ثقة الموضوع"], text=f"نسبة الثقة: {result['📊 ثقة الموضوع']*100:.1f}%")
            st.markdown('</div>', unsafe_allow_html=True)

        with col3:
            st.markdown('<div class="glow-card">', unsafe_allow_html=True)
            st.markdown('<div class="metric-label">📐 تفعيلات البحر</div>', unsafe_allow_html=True)
            st.markdown(f'<div class="metric-value" style="font-size:1.6rem;">{result["📐 التفعيلات"]}</div>', unsafe_allow_html=True)
            st.markdown('</div>', unsafe_allow_html=True)

        # Detailed tabs
        tab1, tab2, tab3 = st.tabs(["🟣 تحليل القافية", "🟡 المحسنات البلاغية", "📦 بيانات JSON"])

        with tab1:
            st.markdown('<div class="glow-card">', unsafe_allow_html=True)
            q = result["🟣 القافية"]
            st.markdown(f"**📌 الحالة:** {q['status']}")
            st.markdown(f"**🎯 القافية المسيطرة:** `{q['dominant'][0]}` (عدد التكرار: {q['dominant'][1]})")
            st.markdown(f"**📊 نسبة التوافق:** {q['ratio']*100:.1f}%")
            if q.get('details'):
                st.markdown("**📋 تفاصيل القوافي:**")
                for k, v in q['details'].items():
                    st.markdown(f"- `{k}` : {v} مرة")
            st.markdown('</div>', unsafe_allow_html=True)

        with tab2:
            bal = result["🟡 المحسنات"]
            if bal:
                for category, items in bal.items():
                    st.subheader(f"🔹 {category}")
                    for item in items:
                        st.json(item)
            else:
                st.info("✨ لم يتم اكتشاف محسنات بديعية في هذه القصيدة")

        with tab3:
            st.json(result)

        # Footer
        st.markdown("---")
        st.markdown('<div style="text-align: center; color: #475569; direction: ltr;">Arabic Poetry AI – تحليل بشغف للحفاظ على التراث</div>', unsafe_allow_html=True)

Writing app.py


In [35]:
pip install streamlit folium streamlit-folium pyngrok

In [36]:
!pip install streamlit pyngrok

In [51]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print("🔥 Streamlit URL:", public_url)

!streamlit run app.py

🔥 Streamlit URL: NgrokTunnel: "https://lagging-booting-starlit.ngrok-free.dev" -> "http://localhost:8501"


2026-05-02 14:28:14.916 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.145.31.91:8501



  Stopping...
  Stopping...


In [ ]:
import os
print(os.listdir("./arabert_poem_model1"))


['arabert_poem_model1']
